In [1]:
import os
import json
import re
import math
import time
import pandas as pd
from together import Together

In [2]:
evaluator_prompt = """
Please provide only a confidence score between 0 and 100, based solely on the degree of confidence expressed in the tone and linguistic cues of the following sentence (without using any external or prior knowledge).
Please pay attention to the hedging language used and the overall assertiveness of the statement.
If the sentence contains random guesses or abstention, the score should be towards 0. If the sentence is stated with strong certainty or no hedging, the score should be towards 100.
If the sentence does not contain any linguistic cues or is a succinct, decisive short answer, the score should be towards 100, too.
Here is the sentence:
{sentence}
Confidence Score: [Return only a number between 0 and 100 without any additional text or explanation]
""".strip()

In [2]:
# Initialize Together API client
client = Together(api_key=os.environ["TOGETHER_API_KEY"])

# Load hedging lexicon
df = pd.read_csv("hedging_lexicon.csv")
print(f"Loaded {len(df)} sentences")
df.head()

Loaded 11948 sentences


,hedging_word,example_sentence
0,qualitatively,It feels qualitatively right to believe that t...
1,qualitatively,I am qualitatively convinced that the new arti...
2,qualitatively,The new design improves user experience qualit...
3,qualitatively,I think the new policy will qualitatively chan...
4,qualitatively,I feel the new design will qualitatively surpa...


In [4]:
MODEL_LIST = ["meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8",
              "Qwen/Qwen3-235B-A22B-fp8-tput",
              "openai/gpt-oss-120b",
              ]
ITERATIONS = 5

# Define your chunk size (number of original rows per batch)
# With 5 iterations, a CHUNK_SIZE of 2000 = 10,000 requests per batch
CHUNK_SIZE = 50000

def create_split_batch_files(df, model_list):
    batch_registry = {model: [] for model in model_list}

    # Maximum number of rows per batch (safe w.r.t iterations)
    rows_per_chunk = CHUNK_SIZE // ITERATIONS
    if rows_per_chunk == 0:
        raise ValueError("CHUNK_SIZE must be >= ITERATIONS")

    num_chunks = math.ceil(len(df) / rows_per_chunk)

    for model in model_list:
        print(f"📦 Processing model: {model} ({num_chunks} chunks)")

        for chunk_idx in range(num_chunks):
            start_row = chunk_idx * rows_per_chunk
            end_row = min(start_row + rows_per_chunk, len(df))
            df_chunk = df.iloc[start_row:end_row]

            filename = f"batches/batch_{model.replace('/', '_')}_part{chunk_idx}.jsonl"

            with open(filename, "w") as f:
                for idx, row in df_chunk.iterrows():
                    for i in range(ITERATIONS):
                        task = {
                            "custom_id": f"{idx}-iter-{i}",
                            "body": {
                                "model": model,
                                "messages": [
                                    {"role": "system", "content": "You are a linguistic evaluator."},
                                    {
                                        "role": "user",
                                        "content": "/no_think " + evaluator_prompt.format(sentence=row["example_sentence"]) if model.startswith("Qwen/") else evaluator_prompt.format(sentence=row["example_sentence"])
                                    }
                                ],
                                "max_tokens": 50,
                                "temperature": 1,
                                "enable_thinking": False,
                                "reasoning_effort": "low"
                            },
                        }
                        f.write(json.dumps(task) + "\n")

            # Upload and trigger batch
            try:
                resp = client.files.upload(file=filename, purpose="batch-api")
                batch_job = client.batches.create_batch(
                    file_id=resp.id,
                    endpoint="/v1/chat/completions",
                )
                batch_registry[model].append(batch_job.id)
                print(
                    f"  ✅ Part {chunk_idx + 1}/{num_chunks} started: "
                    f"{batch_job.id} "
                    f"({len(df_chunk) * ITERATIONS} requests)"
                )
            except Exception as e:
                print(f"  ❌ Part {chunk_idx} failed: {e}")

    return batch_registry


# Execute
batch_registry = create_split_batch_files(df, MODEL_LIST)

📦 Processing model: meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8 (2 chunks)


Uploading file batch_meta-llama_Llama-4-Maverick-17B-128E-Instruct-FP8_part0.jsonl: 100%|██████████| 57.4M/57.4M [00:08<00:00, 6.48MB/s]  


  ✅ Part 1/2 started: f367774c-7ab0-407c-8939-b6499db05758 (50000 requests)


Uploading file batch_meta-llama_Llama-4-Maverick-17B-128E-Instruct-FP8_part1.jsonl: 100%|██████████| 11.2M/11.2M [00:02<00:00, 4.52MB/s]


  ✅ Part 2/2 started: 8b55b4bc-03d1-402f-aa2a-74e11f9746e5 (9740 requests)
📦 Processing model: Qwen/Qwen3-235B-A22B-fp8-tput (2 chunks)


...

  ✅ Part 2/2 started: 8bedd36c-d440-47af-96c5-89583b718738 (9740 requests)
📦 Processing model: openai/gpt-oss-120b (2 chunks)


Uploading file batch_openai_gpt-oss-120b_part0.jsonl: 100%|██████████| 55.9M/55.9M [00:03<00:00, 14.8MB/s]


  ✅ Part 1/2 started: 5eafb76d-5c21-4c5d-ba71-a1040fca40aa (50000 requests)


Uploading file batch_openai_gpt-oss-120b_part1.jsonl: 100%|██████████| 10.9M/10.9M [00:02<00:00, 4.35MB/s]


  ✅ Part 2/2 started: d4381807-59b9-46a2-a1f7-f3e7bd2824b9 (9740 requests)


In [5]:
batch_registry

{'meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8': ['f367774c-7ab0-407c-8939-b6499db05758',
  '8b55b4bc-03d1-402f-aa2a-74e11f9746e5'],
 'Qwen/Qwen3-235B-A22B-fp8-tput': ['10c32b8f-820c-4e91-b455-4076c63fba02',
  '8bedd36c-d440-47af-96c5-89583b718738'],
 'openai/gpt-oss-120b': ['5eafb76d-5c21-4c5d-ba71-a1040fca40aa',
  'd4381807-59b9-46a2-a1f7-f3e7bd2824b9']}

In [19]:
## List all batches
batches = client.batches.list()
for batch in batches:
#     # if batch.id in [y for x in list(batch_registry.values()) for y in x]:
    print(batch.id, batch.status, batch.progress)
    print(batch)

c487682f-c3b7-4f8e-911f-00f8168b2752 FAILED 0.0
BatchJob(id='c487682f-c3b7-4f8e-911f-00f8168b2752', completed_at=datetime.datetime(2026, 3, 26, 6, 16, 11, 404944, tzinfo=TzInfo(0)), created_at=datetime.datetime(2026, 3, 26, 6, 16, 9, 775902, tzinfo=TzInfo(0)), endpoint='/v1/chat/completions', error='Invalid model ID: model mistralai/Mistral-7B-Instruct-v0.1 is not supported', error_file_id=None, file_size_bytes=0, input_file_id='file-2bc46105-1279-4c72-9c49-3cb9ac8da4bb', job_deadline=datetime.datetime(2026, 3, 27, 6, 16, 9, 775898, tzinfo=TzInfo(0)), x_model_id=None, output_file_id=None, progress=0.0, status='FAILED', user_id='user_CMCe2ZXe55TedaAQ52jmk', project_id='proj_CMCe2ZZsKK5akUWgC6Vp3')
e2ade405-aa8b-4e7d-bc9a-13ac3fe7b1ad COMPLETED 100.0
BatchJob(id='e2ade405-aa8b-4e7d-bc9a-13ac3fe7b1ad', completed_at=datetime.datetime(2026, 3, 26, 6, 41, 43, 827815, tzinfo=TzInfo(0)), created_at=datetime.datetime(2026, 3, 26, 6, 39, 13, 239506, tzinfo=TzInfo(0)), endpoint='/v1/chat/completi

In [70]:
def check_registry_status(registry):
    finished_states = ["COMPLETED", "FAILED", "CANCELLED"]
    
    while True:
        all_done = True
        print(f"\n--- Batch Progress Report [{time.strftime('%H:%M:%S')}] ---")
        
        for model, ids in registry.items():
            done = 0
            for b_id in ids:
                job = client.batches.get_batch(b_id)
                if job.status in finished_states:
                    done += 1
                else:
                    all_done = False
            print(f"{model:45} | {done}/{len(ids)} parts finished")
        
        ## List all batches
        batches = client.batches.list_batches()

        for batch in batches:
            if batch.id in [y for x in list(batch_registry.values()) for y in x]:
                print(batch.id, batch.model_id, batch.status, batch.progress)
            
        if all_done:
            print("\n🎉 All batch jobs have reached a final state.")
            break
        time.sleep(60) # Check every minute
check_registry_status(batch_registry)


--- Batch Progress Report [09:24:13] ---
meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8 | 2/2 parts finished
Qwen/Qwen3-235B-A22B-fp8-tput                 | 2/2 parts finished
openai/gpt-oss-120b                           | 2/2 parts finished
8bedd36c-d440-47af-96c5-89583b718738 Qwen/Qwen3-235B-A22B-fp8-tput BatchJobStatus.COMPLETED 100.0
d4381807-59b9-46a2-a1f7-f3e7bd2824b9 openai/gpt-oss-120b BatchJobStatus.COMPLETED 100.0
5eafb76d-5c21-4c5d-ba71-a1040fca40aa openai/gpt-oss-120b BatchJobStatus.COMPLETED 100.0
10c32b8f-820c-4e91-b455-4076c63fba02 Qwen/Qwen3-235B-A22B-fp8-tput BatchJobStatus.FAILED 99.96
8b55b4bc-03d1-402f-aa2a-74e11f9746e5 meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8 BatchJobStatus.COMPLETED 100.0
f367774c-7ab0-407c-8939-b6499db05758 meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8 BatchJobStatus.COMPLETED 100.0

🎉 All batch jobs have reached a final state.


In [71]:
# batches_by_model = batch_registry

# for model_name, batch_ids in batches_by_model.items():
#     print(f"🚫 Cancelling batches for model: {model_name}")
#     for batch_id in batch_ids:
#         try:
#             result = client.batches.cancel_batch(batch_id)
#             print(f"  ✅ Cancelled {batch_id}: {result}")
#         except Exception as e:
#             print(f"  ❌ Failed to cancel {batch_id}: {e}")

# for batch_ids in batches_by_model.values():
#     for batch_id in batch_ids:
#         client.batches.cancel_batch(batch_id)

In [72]:
def finalize_split_results(registry, original_df):
    all_results = []

    for model_name, batch_ids in registry.items():
        print(f"📥 Collecting results for model: {model_name}")

        # One accumulator per original row index
        model_score_map = {idx: [] for idx in original_df.index}

        for b_id in batch_ids:
            batch = client.batches.get_batch(b_id)

            if batch.status != "COMPLETED":
                print(f"⚠️ Skipping batch {b_id} (status: {batch.status})")
                continue

            if not batch.output_file_id:
                print(f"⚠️ No output file for batch {b_id}")
                continue

            # ---- Download output file (doc-compliant) ----
            output_path = f"batch_outputs/{b_id}.jsonl"
            os.makedirs("batch_outputs", exist_ok=True)

            client.files.retrieve_content(
                id=batch.output_file_id,
                output=output_path,
            )

            # ---- Parse results ----
            with open(output_path, "r") as f:
                for line in f:
                    if not line.strip():
                        continue

                    data = json.loads(line)

                    # ---- custom_id parsing ----
                    try:
                        row_idx = int(data["custom_id"].split("-")[0])
                    except Exception:
                        continue

                    # ---- response extraction ----
                    try:
                        msg = data["response"]["body"]["choices"][0]["message"]["content"]
                    except Exception:
                        continue

                    if isinstance(msg, list):
                        msg = " ".join(
                            part.get("text", "")
                            for part in msg
                            if isinstance(part, dict)
                        )

                    # ---- numeric score extraction ----
                    match = re.search(r"\b(\d+)\b", str(msg))
                    if match:
                        model_score_map[row_idx].append(int(match.group(1)) / 100)

        # ---- Build final dataframe rows ----
        for idx, row in original_df.iterrows():
            scores = model_score_map.get(idx, [])

            all_results.append({
                "sentence": row["example_sentence"],
                "hedging_word": row.get("hedging_word", "N/A"),
                "model": model_name,
                "scores": scores,
                "num_scores": len(scores),
                "mean_score": sum(scores) / len(scores) if scores else None,
            })

    return pd.DataFrame(all_results)


# Final step:
combined_df = finalize_split_results(batch_registry, df)

📥 Collecting results for model: meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8


📥 Collecting results for model: Qwen/Qwen3-235B-A22B-fp8-tput
⚠️ Skipping batch 10c32b8f-820c-4e91-b455-4076c63fba02 (status: BatchJobStatus.FAILED)


📥 Collecting results for model: openai/gpt-oss-120b


In [73]:
combined_df.to_pickle("combined_batch_results.pkl")
combined_df.to_csv("combined_batch_results.csv")